# 11 · A different pattern: encounter with a relative 🧬

> **Note**
>
> This notebook is **best run locally** — the pattern needs a few thousand small
> time steps to grow (a progress bar tells you how far along it is).

![The rainbow Beast (left) beside its Turing-patterned relative (right)](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/turing.jpg)

In 1952 Alan **Turing** asked a startling question: how does a featureless ball of
cells — an early embryo — decide *where* to put spots, stripes, fingers? His answer
was **reaction–diffusion**. Two substances, an **activator** and an **inhibitor**,
react with each other and diffuse at **different speeds**; that imbalance can make a
perfectly uniform state spontaneously break up into a regular pattern. The same
mechanism is thought to paint **leopard spots, zebra stripes, seashells and fish** —
*morphogenesis*, pattern from no pattern.

So where did **the beast** get its rainbow coat? We put Turing's mechanism on the
**surface of the sculpture itself** — a genuine **PDE on a curved 2-manifold** — and
watch it grow its own skin. This is where Part II's pillars **fuse**: an **unsteady**
problem (the time stepping of unit 8) whose reaction is **nonlinear** (the Newton idea of
unit 9), now a *coupled* pair of species — plus one new ingredient, **surface finite
elements**, all written the NGSolve way, **variationally**.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from netgen.occ import OCCGeometry, Sphere, Cylinder, Glue, Pnt, X, Y, Z
from netgen.meshing import MeshingStep
from ngsolve import *
from ngsolve.webgui import Draw
import sys

In [ ]:
def progress(i, n):                                    # a tiny dependency-free bar (live only)
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent (no \r spam in the HTML)
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:  # (survives JupyterLite/Colab/local)
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  growing the coat… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. The beast's skin — a surface mesh

The beast is the **solid sculpture** of notebook 2 (a spherical shell bored by three
cylinders). We want a mesh of its **surface only** — a 2-D manifold in 3-D. **Two routes:**

1. **Faces → mesh** (used here): keep the solid's `faces`, `Glue` them into a closed
   **shell**, and mesh *that*.
2. **Solid → stop at the surface**: mesh the solid but halt after the surface step with
   `perfstepsend=MeshingStep.MESHSURFACE` — no volume mesh is ever built.

Both yield the same surface mesh. On it the measure is **`ds`** (surface area). We keep the
mesh **coarse** and resolve the pattern by **high order** ($k=4$) instead of tiny triangles —
fewer, bigger curved elements, each carrying a quartic field.

> **Why `grad(u).Trace()`?**
>
> On a surface mesh the elements are **boundary** elements, so a plain `grad(u)` is invalid
> inside a `ds`-form — take its **tangential trace** `grad(u).Trace()` (the surface gradient
> $\\nabla_\\Gamma u$). Without it, assembly aborts: *"Trialfunction does not support BND-forms,
> maybe a Trace() operator is [missing]"*.

In [ ]:
def beast_sculpture():
    s = Sphere(Pnt(50, 50, 50), 80) - Sphere(Pnt(50, 50, 50), 50)
    for p, d in [(Pnt(-100, 0, 0), X), (Pnt(100, -100, 100), Y), (Pnt(0, 100, -100), Z)]:
        s = s - Cylinder(p, d, r=40, h=300)
    return s.Move((-50, -50, -50)).Scale(Pnt(0, 0, 0), 0.05)     # centre + shrink

solid = beast_sculpture()

# Route 1 — glue the faces into a closed shell, mesh the surface:
shell = Glue([f for f in solid.faces])
mesh = Mesh(OCCGeometry(shell).GenerateMesh(maxh=0.75))          # coarse — high order resolves the pattern
mesh.Curve(3)

# Route 2 (alternative) — mesh the solid but STOP after surface meshing (no volume elements):
mesh_alt = Mesh(OCCGeometry(solid).GenerateMesh(maxh=0.75, perfstepsend=MeshingStep.MESHSURFACE))

print(f"surface mesh: {mesh.nv} vertices, area = {Integrate(CF(1)*ds, mesh):.1f}  "
      f"(route 2 agrees: {mesh_alt.nv} vertices)")
Draw(mesh)

## 2. The model — an activator and an inhibitor

The **Gray–Scott** system for two surface concentrations $u$ (substrate) and $v$ (activator):
$$ \partial_t u = D_u\,\Delta_\Gamma u - u v^2 + F(1-u),\qquad
   \partial_t v = D_v\,\Delta_\Gamma v + u v^2 - (F+k)\,v . $$

- $v$ is **autocatalytic**: $uv^2$ converts substrate $u$ into *more* $v$.
- $F$ **feeds** fresh $u$; $k$ **removes** $v$.
- The activator diffuses **slower** than the substrate ($D_v<D_u$) — *short-range
  activation, long-range inhibition*, exactly Turing's recipe.
- Tune $F,k$ for **dots ↔ stripes ↔ coral**; $\Delta_\Gamma$ is the **surface** Laplacian.

The whole pipeline at a glance (**click to enlarge**):

![Gray-Scott pipeline: surface, equations, seeding, time stepping, emerging pattern](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/gray-scott-model.webp)

In [ ]:
fes = H1(mesh, order=4)            # high order k=4 on the coarse mesh (p-refinement)
u, w = fes.TnT()
M = BilinearForm(u * w * ds, check_unused=False).Assemble()                    # surface mass
K = BilinearForm(grad(u).Trace() * grad(w).Trace() * ds, check_unused=False).Assemble()  # surface stiffness

Du, Dv, F, k = 3.6e-3, 1.8e-3, 0.037, 0.060        # "coral" regime; note D_v = D_u/2
dt = 1.0

## 3. A variational semi-implicit (IMEX) scheme

Diffusion is **stiff** (it couples the whole surface), so we take it **implicitly**;
the local **reaction** stays **explicit**. Each species then needs one pre-factorised
solve per step, $M+\Delta t\,D\,K$ — symmetric positive definite, so `sparsecholesky`
fits. The reaction is a **nonlinear form** we never assemble: we just **`Apply`** it
to the current state each step (the same trick as the DG transport in notebook 12) —
no hand-indexing of coefficients, everything stays variational.

In [ ]:
def factor(D):
    mstar = M.mat.CreateMatrix()
    mstar.AsVector().data = M.mat.AsVector() + dt * D * K.mat.AsVector()
    return mstar.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

inv_u, inv_v = factor(Du), factor(Dv)

gfu, gfv = GridFunction(fes), GridFunction(fes)
react_u = BilinearForm(fes, nonassemble=True)
react_u += -(-u * gfv * gfv + F * (1 - u)) * w * ds        # r_u(u, v) tested, as an operator
react_v = BilinearForm(fes, nonassemble=True)
react_v += -(gfu * u * u - (F + k) * u) * w * ds           # r_v(u, v); here `u` is the v-field

## 4. Seed the beast and let the pattern grow

We start from a calm skin ($u=1$, $v=0$) and dab a few **patches** of activator onto
the surface. The initial fields are set by an honest **$L^2$-projection** (solve
$M\,\mathbf c = \int (\cdot)\,w\,ds$) — no poking at raw arrays. From those seeds,
spots bud, split and creep across the whole beast: a chemical coral reef.

In [ ]:
# A few smooth activator dabs. On a surface mesh the elements ARE boundary elements, so the
# fields are set with **definedon=mesh.Boundaries(".*")** (an L2-projection onto the surface).
seed = sum(exp(-((x - px)**2 + (y - py)**2 + (z - pz)**2) / 2.0)
           for (px, py, pz) in [(4, 0, 0), (0, 4, 0), (0, 0, 4),
                                (-4, 0, 0), (0, -4, 0), (0, 0, -4)])
gfu.Set(1 - 0.5 * seed, definedon=mesh.Boundaries(".*"))    # substrate u
gfv.Set(0.25 * seed,    definedon=mesh.Boundaries(".*"))    # activator v

res = gfu.vec.CreateVector()
nsteps = 6500
growth = GridFunction(fes, multidim=0)                      # activator v snapshots, at i * nsteps / 4
growth_u = GridFunction(fes, multidim=0)                    # substrate u snapshots, same times
growth.AddMultiDimComponent(gfv.vec); growth_u.AddMultiDimComponent(gfu.vec)   # i = 0: the bare seed
snap_at = {round(i * nsteps / 4) for i in (1, 2, 3)}        # i = 1, 2, 3 (mind the rounding)
with TaskManager():
    for step in range(nsteps):
        react_u.Apply(gfu.vec, res); gfu.vec.data = inv_u * (M.mat * gfu.vec - dt * res).Evaluate()
        react_v.Apply(gfv.vec, res); gfv.vec.data = inv_v * (M.mat * gfv.vec - dt * res).Evaluate()
        if step + 1 in snap_at:
            growth.AddMultiDimComponent(gfv.vec); growth_u.AddMultiDimComponent(gfu.vec)
        progress(step, nsteps)

The coat is best seen **growing** — a webgui animation of four snapshots, from the bare seed
to the full labyrinth (press ▶; it plays slowly, drag to rotate). This single scene replaces a
separate still. `order=3` draws the smooth high-order field, the colour range is clamped to the
activator's band **[0, 0.3]** (hiding the high-order over/undershoot at the sharp pattern edges),
and the mesh wireframe / edges are off.

In [ ]:
no_grid = {"Objects": {"Wireframe": False, "Edges": False},
           "Multidim": {"speed": 0.3, "animate": True}}     # play the growth slowly
Draw(growth, mesh, "v growing (press ▶)", order=3, min=0, max=0.3, autoscale=False,
     interpolate_multidim=True, animate=True, settings=no_grid)

The **substrate** $u$ tells the inverse story — it is **consumed** wherever the activator
blooms, so its labyrinth is the photographic negative of the coat above.

In [ ]:
Draw(growth_u, mesh, "u — the substrate (press ▶)", order=3, min=0, max=1, autoscale=False,
     interpolate_multidim=True, animate=True, settings=no_grid)

From a few dabs of activator, the beast grows a full coat of **labyrinth stripes**,
entirely on its own — no pattern was ever prescribed, only two reacting, differently-
diffusing chemicals. Both species, side by side — the activator $v$ in the warm **coat** palette
beside the substrate $u$ in a cool **ice** palette (its photographic negative), each Beast
spinning about its own axis — rendered offline (`scripts/render_turing_video.py`):

![Two Beasts side by side — the activator's labyrinth in warm browns and the substrate's inverse pattern in cool teal, both spinning](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/turing.gif)

Strictly, this striped creature is not *the* rainbow Beast but a **close relative** — the
same sculpture, wearing a coat it **grew** rather than one we painted on. We hereby claim
it for the Beast's family. 🧬

## Supplementary — VTK output for ParaView, PyVista & Blender

That offline render went through **PyVista**. The bridge from NGSolve to *any* external tool
(ParaView, PyVista, Blender) is a **VTK** file — the mesh plus one or more coefficient functions,
written in a single call:

```python
vtk = VTKOutput(mesh, coefs=[gfv, gfu], names=["activator", "substrate"],
                filename="turing", subdivision=2)
vtk.Do()                  # -> turing.vtu ; open in ParaView, or pyvista.read("turing.vtu")
```

A **high-order** field must be sampled into the (cell-wise) VTK file — two knobs:

- **`subdivision=N`** — split each element into $2^N$ pieces and sample the field **linearly** at
  the sub-vertices: more, smaller **linear** cells, readable by *every* VTK tool.
- **`order=M`** *(new)* — write native **higher-order** (Lagrange) VTK cells of order $M$: far
  fewer cells, exact to order $M$ — but needs a recent reader (ParaView ≥ 5.5).

Use `subdivision` for portability, `order` for compact high-order fidelity. Then **ParaView** to
explore, **PyVista** for scripted renders (as in `render_turing_video.py`), and **Blender** (via
VTK / `.x3d` import) for cinematic shading.

**Next:** two more couplings before the trail's end — warming the chocolate bar until it
**bends** (notebook 16), and finally letting the chocolate **melt** (notebook 17).

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("10-dg-hdg", "10 · Discontinuous Galerkin & HDG")
    _next = ("12-pedestrian-dynamics", "12 · A crowd heads for coffee 🚶☕")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))